# GeoAI — Etkileşimli Kullanım Defteri

Bu defter GeoAI boru hattını adım adım gösterir:

1. Sentetik veriyle hızlı demo
2. Sonuç tablosunu ve öncelik sınıflarını inceleme
3. Fizik tabanlı gerçekçi test verisini **dosyalara** yazıp dosyalardan yükleyerek uçtan uca çalıştırma
4. Dışa aktarılan grid dosyalarını inceleme
5. Kendi verinizle çalışma şablonu

Defteri depo kök klasöründen açın (`jupyter notebook geoai_notebook.ipynb`).

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('.'))   # depo kökünden çalıştırıldığında geoai paketini bul

import pandas as pd

import geoai
from geoai.pipeline import GeoAIPipeline

print('GeoAI sürümü:', geoai.__version__)

## 1. Demo (sentetik veri)

`show_plots=True` Jupyter içinde grafikleri gösterir; `save_plots=True` PNG olarak da kaydeder.

In [ ]:
results = GeoAIPipeline.run_demo(
    nx=100, ny=100,
    target_types=['mineral', 'groundwater'],
    output_dir='output_notebook/demo',
    show_plots=True,
    save_plots=True,
)

## 2. Hedef tablosu

`priority` sütunu teknik rapordaki karar kuralıdır:
Öncelik-1: P ≥ 0.75 ve U ≤ 0.20 · Öncelik-2: P ≥ 0.60 ve U ≤ 0.30 · Öncelik-3: diğer.

In [ ]:
from geoai.utils.reporting import combine_targets

table = combine_targets({t: r['targets'] for t, r in results.items()})
table[['target_type', 'rank', 'priority', 'x', 'y', 'max_probability',
       'mean_uncertainty', 'area_km2']].head(15)

In [ ]:
# Model performansı (kuyu-gruplu uzaysal cross-validation, out-of-fold)
for typ, res in results.items():
    print(typ, res['mode'], res['cv_info'].get('strategy'), res['cv_info'].get('n_splits'), 'fold')
    display(pd.DataFrame(res['cv_results']).T)

## 3. Gerçekçi test verisi: dosyaya yaz → dosyadan yükle

`generate_test_data.py` potansiyel alan fiziğiyle (dipol TMI, küre/prizma gravitesi) katmanlar ve
dengesiz kuyu verisi üretir. Katmanlar farklı çözünürlük ve hizalamalarla kaydedilir; böylece
co-registration da test edilir.

In [ ]:
from generate_test_data import RealisticGeoDataGenerator, save_test_dataset

gen = RealisticGeoDataGenerator(nx=120, ny=120, scenario='porphyry_copper', seed=7)
layers = gen.generate()
wells = gen.generate_wells(imbalance_ratio=8.0, n_total=60)
data_dir = 'output_notebook/realistic_data'
files = save_test_dataset(layers, wells, gen, data_dir, formats=['surfer_grd'])
files

In [ ]:
pipe = GeoAIPipeline(project_name='Porfiri_Test')
for path in files:
    if path.endswith('.grd'):
        pipe.add_layer(path)          # katman tipi dosya adından tespit edilir
pipe.add_wells(os.path.join(data_dir, 'wells.csv'))   # TARGET_TYPE kolonuna göre ayrılır

real_results = pipe.run(
    target_types=['mineral'],
    target_nx=120, target_ny=120,
    min_prob=None,                    # CV ile bulunan optimal eşik
    output_dir='output_notebook/realistic',
    show_plots=True, save_plots=True,
)

In [ ]:
# Bilinen (sentetik) hedef zonlarına uzaklık
import numpy as np

for zx, zy, zr, kind in gen._target_zones:
    d = [np.hypot(t['x'] - zx, t['y'] - zy) for t in real_results['mineral']['targets']]
    print(f"{kind:10} zon ({zx:.0f}, {zy:.0f}) → en yakın hedef: {min(d) if d else float('nan'):.0f} m")

## 4. Dışa aktarılan grid dosyaları

Olasılık (P), belirsizlik (U) ve güven skoru (S) haritaları `grids/` altına Surfer GRD ve
(rasterio varsa) GeoTIFF olarak yazılır. Bunlar QGIS, ArcGIS, Surfer veya Oasis Montaj'da açılabilir.

In [ ]:
from geoai.cli import describe_grid

print(describe_grid('output_notebook/realistic/grids/mineral_probability.grd'))

## 5. Kendi verinizle

Aşağıdaki hücre bir şablondur; dosya yollarını kendi verinize göre düzenleyin.
Desteklenen formatlar: Surfer GRD (ASCII/6/7), Geosoft GRD, ER Mapper ERS, ESRI ASCII (.asc),
GeoTIFF ve (rasterio ile) GDAL'ın okuduğu diğer rasterlar, XYZ/CSV/DAT.

In [ ]:
KATMANLAR = {
    # 'magnetic_tmi': 'data/magnetic.grd',
    # 'gravity_bouguer': 'data/bouguer.grd',
    # 'resistivity': 'data/resistivity.tif',
}
KUYU_CSV = None   # 'data/wells.csv'  (X, Y, LABEL [, TARGET_TYPE])

if KATMANLAR:
    proje = GeoAIPipeline(project_name='Benim_Projem')
    for isim, yol in KATMANLAR.items():
        proje.add_layer(yol, name=isim)
    if KUYU_CSV:
        proje.add_wells(KUYU_CSV)
    sonuc = proje.run(target_types=['mineral'], target_nx=300, target_ny=300,
                      min_prob=None, output_dir='output_notebook/proje')
else:
    print('KATMANLAR boş — şablonu düzenleyin.')